<a href="https://colab.research.google.com/github/latidore/Genesis/blob/main/FTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [98]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
import ipywidgets as widgets

# 1. Define the core mathematical functions globally
def F(x):
    """The antiderivative function F(x)."""
    return (x - 2) * x * (x + 1) + 1

def smallf(x):
    """The derivative function f(x) = F'(x)."""
    return 3 * x**2 - 2 * x - 2

def _calculate_parameters(n, a_ref, b_ref):
    """Calculates step size and F(a), F(b) values."""
    step = (b_ref - a_ref) / n
    if step <= 0:
        step = 0.01
    Fa_ref = F(a_ref)
    Fb_ref = F(b_ref)
    return step, Fa_ref, Fb_ref

def _generate_plot_data(F, smallf, a_ref, b_ref, step):
    """Generates all x and y coordinates needed for plotting."""
    x_values_segments = np.arange(a_ref, b_ref + step, step)
    x_smooth = np.linspace(a_ref - 0.5, b_ref + 0.5, 200)
    y_smooth = F(x_smooth)
    return x_values_segments, x_smooth, y_smooth

def _plot_F_x_graph(ax1, F, smallf, a_ref, b_ref, Fa_ref, Fb_ref, step, x_values_segments, x_smooth, y_smooth, n):
    """Plots the F(x) graph with approximations on ax1."""
    # F(x) 그래프 그리기 (전체는 약간 희미하게)
    ax1.plot(x_smooth, y_smooth, color='gray', alpha=0.3, linewidth=2, label='$F(x)$')

    # (a, F(a)) 에서 (b, F(b)) 까지의 graph 를 좀 더 굵게
    x_highlight = x_smooth[(x_smooth >= a_ref) & (x_smooth <= b_ref)]
    y_highlight = F(x_highlight)
    ax1.plot(x_highlight, y_highlight, color='gray', alpha=0.6, linewidth=3, label='F(x) Segment')

    # F'(x)dx 시각화 (접선 방향 진행)
    for x_i in x_values_segments[:-1]:
        y_i = F(x_i)
        slope = smallf(x_i)
        dy_approx = slope * step
        next_x = x_i + step
        next_y_approx = y_i + dy_approx

        color = 'red' if dy_approx > 0 else 'blue'

        ax1.plot([x_i, next_x], [y_i, y_i], color='black', alpha=0.3, linewidth=1) # Horizontal dx
        ax1.plot([next_x, next_x], [y_i, next_y_approx], color=color, linewidth=2) # Vertical F'(x)dx
        ax1.plot([x_i, next_x], [y_i, next_y_approx], color=color, linestyle='--', alpha=0.6) # Tangent line

    # Draw faint vertical lines at each dx interval
    for x_interval in np.arange(a_ref, b_ref + step, step):
        ax1.vlines(x_interval, ax1.get_ylim()[0], ax1.get_ylim()[1], colors='gray', linestyles='--', linewidth=0.5, alpha=0.2)

    # Plot points at (a, F(a)) and (b, F(b))
    ax1.plot(a_ref, Fa_ref, 'o', color='darkgreen', markersize=8, label='Point (a, F(a))')
    ax1.plot(b_ref, Fb_ref, 'o', color='darkmagenta', markersize=8, label='Point (b, F(b))')

    # Axis markers and labels
    ax1.vlines(a_ref, -4, Fa_ref, colors='green', linestyles=':', linewidth=1.5)
    ax1.vlines(b_ref, -4, Fb_ref, colors='purple', linestyles=':', linewidth=1.5)
    ax1.hlines(Fa_ref, -6, a_ref, colors='green', linestyles=':', linewidth=1.5)
    ax1.hlines(Fb_ref, -6, b_ref, colors='purple', linestyles=':', linewidth=1.5)
    ax1.text(a_ref, -3.0, f'a = {a_ref:.2f}', color='green', ha='center', fontweight='bold', fontsize=12)
    ax1.text(b_ref, -3.0, f'b = {b_ref:.2f}', color='purple', ha='center', fontweight='bold', fontsize=12)
    ax1.text(-1.5, Fa_ref, f'$F(a) = {Fa_ref:.2f}$', color='green', va='center', fontweight='bold', fontsize=12)
    ax1.text(-1.5, Fb_ref, f'$F(b) = {Fb_ref:.2f}$', color='purple', va='center', fontweight='bold', fontsize=12)

    # Vertical Bounding Segment
    ax1.plot([a_ref, a_ref], [Fa_ref, Fb_ref], color='magenta', linewidth=3, label='Vertical Bounding Segment')

    # Graph limits, title, grid, axes
    ax1.set_xlim(a_ref - 1.5, b_ref + 1)
    ax1.set_ylim(min(F(x_smooth)) - 0.5, max(F(x_smooth)) + 0.5)
    ax1.set_title(f'$F(x) = (x-2)x(x+1)+1$ (n: {n}, dx: {step:.2f})')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color='black', linewidth=0.8)
    ax1.axvline(0, color='black', linewidth=0.8)

def _plot_smallf_x_bars(ax2, smallf, step, x_values_segments):
    """Plots the f(x)/dx bars on ax2."""
    x_bar_centers = x_values_segments[:-1] + step / 2
    f_x_values = smallf(x_values_segments[:-1])
    bar_colors = ['red' if val > 0 else 'blue' for val in f_x_values]

    ax2.bar(x_bar_centers, f_x_values, width=step, color=bar_colors, alpha=0.7, align='center')
    ax2.set_title('$f(x) = 3x^2 - 2x - 2$')
    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.grid(True, alpha=0.3)

    # Calculate the sum of the areas (f(x) * dx)
    total_sum_approximation = np.sum(f_x_values * step)
    text_to_display = r'$\sum f(x_i) \Delta x \approx \int_a^b f(x)dx = ' + f'{total_sum_approximation:.2f}$'
    ax2.text(0.95, 0.95, text_to_display,
             transform=ax2.transAxes, ha='right', va='top', fontsize=12,
             bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5))

def plot_FTC_notation(n=20):
    """Main function to plot the Fundamental Theorem of Calculus visualization."""
    a_ref = -1.0
    b_ref = 3.0

    step, Fa_ref, Fb_ref = _calculate_parameters(n, a_ref, b_ref)
    x_values_segments, x_smooth, y_smooth = _generate_plot_data(F, smallf, a_ref, b_ref, step)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
    # Separating plain text and MathText for suptitle for robust rendering
    # The previous attempt caused ParseException because the entire string was parsed as MathText,
    # and the plain text part before the first '$' was not valid MathText.
    # To fix this, we enclose the plain text part within \text{} inside the MathText block.
    fig.suptitle(r"$\text{Fundamental Theorem of Calculus: } F(x)|_{a}^{b} = \int_{a}^{b} f(x)dx$", fontsize=16)

    _plot_F_x_graph(ax1, F, smallf, a_ref, b_ref, Fa_ref, Fb_ref, step, x_values_segments, x_smooth, y_smooth, n)
    _plot_smallf_x_bars(ax2, smallf, step, x_values_segments)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make space for suptitle

interact(plot_FTC_notation,
         n=widgets.IntSlider(min=1, max=100, step=1, value=20, description='Number of Divisions (n)'));

interactive(children=(IntSlider(value=20, description='Number of Divisions (n)', min=1), Output()), _dom_class…